# Lesson 01 - Python & Jupyter Basics

## Objectives
By the end of this lesson, you will be able to:
- Use core Python syntax (variables, types, loops, conditionals, functions).
- Work effectively in a Jupyter notebook workflow.
- Apply basic exception handling and logging patterns.
- Run a tiny end-to-end data task from a public web dataset.

## Environment Check

This notebook expects `numpy`, `pandas`, `matplotlib`, `seaborn`, `jupyter`, and `ipykernel`.

If packages are missing, install from the repo root:

```bash
pip install -r requirements.txt
```

Then restart the kernel and run all cells in order.

In [ ]:
import sys

print(f"Python version: {sys.version}")

## Python Basics: Variables, Types, and Strings

In [ ]:
age = 28
height_m = 1.75
name = "Ahmad"
is_learning_ai = True

print(type(age), type(height_m), type(name), type(is_learning_ai))
print(f"{name} is {age} years old and is learning AI: {is_learning_ai}")

# Simple arithmetic
score_a = 88
score_b = 92
avg_score = (score_a + score_b) / 2
print(f"Average score: {avg_score}")

## Python Basics: Lists, Dictionaries, Loops, and Conditionals

In [ ]:
temperatures = [30.5, 31.2, 29.8, 33.1]
student_scores = {"Ali": 88, "Sara": 95, "Noor": 79}

# Loop example
for t in temperatures:
    if t >= 32:
        print(f"Hot day: {t}")

# Dictionary iteration
for student, score in student_scores.items():
    status = "pass" if score >= 80 else "needs improvement"
    print(f"{student}: {score} ({status})")

# Small exercise: compute max and min
print(f"Max temperature: {max(temperatures)}")
print(f"Min temperature: {min(temperatures)}")

## Functions and Modular Code

In [ ]:
from typing import Iterable


def compute_bmi(weight_kg: float, height_m: float) -> float:
    """Compute Body Mass Index (BMI).

    Args:
        weight_kg: Weight in kilograms.
        height_m: Height in meters.

    Returns:
        BMI value as float.

    Raises:
        ValueError: If `height_m` is not positive.
    """
    if height_m <= 0:
        raise ValueError("height_m must be > 0")
    return weight_kg / (height_m ** 2)


def normalize_scores(scores: Iterable[float]) -> list[float]:
    """Normalize scores to [0, 1] range using min-max normalization."""
    scores = list(scores)
    if not scores:
        return []
    min_s = min(scores)
    max_s = max(scores)
    if min_s == max_s:
        return [0.5 for _ in scores]
    return [(s - min_s) / (max_s - min_s) for s in scores]

print(f"BMI: {compute_bmi(70, 1.75):.2f}")
print(normalize_scores([50, 70, 90]))

## Error Handling and Logging (Business Exceptions)

In [ ]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s"
)
logger = logging.getLogger("lesson01")


class DataValidationError(Exception):
    """Raised when input data fails business validation."""


def parse_positive_float(value: str) -> float:
    """Parse and validate positive numeric input."""
    try:
        number = float(value)
    except ValueError as exc:
        logger.error("Input is not numeric: %s", value)
        raise ValueError("Input must be numeric") from exc

    if number <= 0:
        logger.warning("Non-positive value rejected: %s", number)
        raise DataValidationError("Value must be positive")

    logger.info("Validated input: %.2f", number)
    return number

for raw in ["3.5", "-2", "abc"]:
    try:
        parsed = parse_positive_float(raw)
        print(f"Accepted: {parsed}")
    except (ValueError, DataValidationError) as exc:
        print(f"Rejected input '{raw}': {exc}")

## Mini End-to-End Example: Load Public Dataset, Transform, Aggregate

In [ ]:
import pandas as pd

TIPS_CSV_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv"

try:
    df = pd.read_csv(TIPS_CSV_URL)
    logger.info("Loaded dataset shape: %s", df.shape)
except Exception as exc:  # network, HTTP, parser-level failures
    logger.exception("Failed to download dataset from %s", TIPS_CSV_URL)
    raise

required_cols = {"total_bill", "tip", "day"}
missing = required_cols.difference(df.columns)
if missing:
    logger.error("Missing required columns: %s", sorted(missing))
    raise KeyError(f"Missing required columns: {sorted(missing)}")

# Simple transformation
df["tip_pct"] = (df["tip"] / df["total_bill"]) * 100

# Simple filtering
high_bill_df = df[df["total_bill"] >= 20]

# Aggregation
summary = (
    high_bill_df.groupby("day", as_index=False)
    .agg(avg_total_bill=("total_bill", "mean"), avg_tip_pct=("tip_pct", "mean"), n_rows=("day", "count"))
    .sort_values("avg_total_bill", ascending=False)
)

print("Top rows from raw dataset:")
print(df.head(3))
print("\nAggregated summary for bills >= 20:")
print(summary)

## Business Logic & Exceptions

In production AI systems, these same patterns are everywhere:

- **Functions and modular code** keep data ingestion, validation, transformation, and model inference independently testable.
- **Input validation + exceptions** protect model endpoints from malformed payloads, missing fields, and invalid value ranges.
- **Logging** provides observability for debugging failures, tracking data-quality incidents, and auditing pipeline behavior.

Example scenario: an event pipeline ingests user interactions for a recommendation system.
If `user_id` is missing or numeric fields are corrupted, validation should raise explicit exceptions and route records to a dead-letter queue instead of silently failing. Logs at warning/error levels let operators detect spikes in bad events before model quality degrades.

## Interview Questions & Answers

1. **Q: Why is logging important in AI/ML pipelines?**  
   **A:** It gives traceability for data issues, model failures, latency spikes, and downstream integration errors.

2. **Q: How would you handle invalid input to a model in production?**  
   **A:** Validate schema and value ranges early, reject or quarantine bad records, and log structured error context.

3. **Q: Difference between syntax errors and runtime exceptions in Python?**  
   **A:** Syntax errors are parse-time issues before execution; runtime exceptions happen while code is running.

4. **Q: Why use Jupyter for AI work?**  
   **A:** It combines narrative + executable code + outputs for fast experimentation and reproducible analysis.

5. **Q: What is the benefit of type hints in Python?**  
   **A:** They improve readability, editor/tooling support, and reduce interface ambiguity across modules.

6. **Q: When should you create a custom exception class?**  
   **A:** When domain-specific failure modes need explicit handling (e.g., business validation failures).

7. **Q: Why write small functions instead of long scripts?**  
   **A:** Small functions are easier to test, debug, reuse, and compose in production pipelines.

8. **Q: What notebook anti-pattern can cause hidden bugs?**  
   **A:** Running cells out of order, which creates hidden state and non-reproducible results.

9. **Q: How do basic Python skills translate to AI engineering?**  
   **A:** They underpin data processing, feature logic, error handling, API orchestration, and model serving glue code.
